# Exploração dos Dados — Olist E-Commerce

Este notebook realiza a exploração inicial dos 7 CSVs utilizados no projeto.
O objetivo é entender a estrutura de cada tabela antes de qualquer transformação:
volume de linhas, tipos de dados, presença de nulos e duplicatas.

Nenhum tratamento é aplicado aqui — as decisões tomadas durante a exploração
são implementadas no notebook `02_tratamento.ipynb`.

**Tabelas exploradas:**
- `olist_customers_dataset` — clientes
- `olist_orders_dataset` — pedidos
- `olist_order_items_dataset` — itens dos pedidos
- `olist_order_reviews_dataset` — avaliações
- `olist_order_payments_dataset` — pagamentos
- `olist_products_dataset` — produtos
- `olist_sellers_dataset` — vendedores

In [1]:
import pandas as pd

## 1. Clientes (`olist_customers_dataset`)

Contém informações dos clientes: identificador único, cidade e estado.
Importante: `customer_id` é gerado por pedido — o mesmo cliente pode ter múltiplos `customer_id`.
O identificador real do cliente é `customer_unique_id`.

In [2]:
df_consumidores = pd.read_csv('../data/raw/olist_customers_dataset.csv')
df_consumidores.shape

(99441, 5)

In [3]:
df_consumidores.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB


In [4]:
df_consumidores.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [5]:
df_consumidores.duplicated().sum()

np.int64(0)

In [6]:
df_consumidores.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


**Conclusão:** A tabela não apresenta valores nulos nem duplicatas. Tipos de dados corretos.
Nenhum tratamento necessário — apenas `customer_state` será utilizado no modelo dimensional.

## 2. Pedidos (`olist_orders_dataset`)

Tabela central do modelo. Contém o ciclo de vida de cada pedido: datas de compra,
aprovação, despacho e entrega. As colunas de data são lidas como string pelo Pandas
e precisarão de conversão no tratamento.

In [7]:
df_compras = pd.read_csv('../data/raw/olist_orders_dataset.csv')
df_compras.shape

(99441, 8)

In [8]:
df_compras.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


In [9]:
df_compras.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [10]:
df_compras.duplicated().sum()

np.int64(0)

In [11]:
df_compras['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

**Conclusão:** Nulos presentes nas colunas de data de entrega — esperado para pedidos
cancelados ou ainda em trânsito. As colunas de data precisam de conversão para datetime.
`time_delivered` será calculado apenas para pedidos com `order_status == 'delivered'`.

## 3. Itens dos Pedidos (`olist_order_items_dataset`)

Contém um registro por item dentro de cada pedido. Um pedido pode ter múltiplos itens
de categorias e vendedores diferentes — granularidade de item, não de pedido.

In [12]:
df_itens = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
df_itens.shape

(112650, 7)

In [13]:
df_itens.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 18.4 MB


In [14]:
df_itens.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [15]:
df_itens.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


**Conclusão:** Sem nulos. `order_id` é a chave de ligação com `olist_orders_dataset`.
No modelo dimensional, `order_id` será mantido como dimensão degenerada em `f_order_items`.

## 4. Avaliações (`olist_order_reviews_dataset`)

Contém as notas e comentários dos clientes por pedido. Um pedido pode ter mais de uma
avaliação registrada — no tratamento, será calculada a média por `order_id`.

In [16]:
df_avaliacoes = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
df_avaliacoes.shape

(99224, 7)

In [17]:
df_avaliacoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 17.8 MB


In [18]:
df_avaliacoes.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [19]:
# Verificando pedidos com mais de uma avaliação
df_avaliacoes.groupby('order_id').size().value_counts()

1    98126
2      543
3        4
Name: count, dtype: int64

**Conclusão:** Existem pedidos com mais de uma avaliação. No tratamento, será calculada
a média de `review_score` por `order_id` para evitar duplicação de linhas no merge.

## 5. Pagamentos (`olist_order_payments_dataset`)

Contém um registro por transação de pagamento. Um pedido pode ser pago com múltiplos
métodos — por exemplo, parte em cartão e parte em voucher. Granularidade de transação.

In [20]:
df_pagamento = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
df_pagamento.shape

(103886, 5)

In [21]:
df_pagamento.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB


In [22]:
df_pagamento.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [23]:
df_pagamento['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [24]:
# Verificando pedidos com múltiplos métodos de pagamento
pedidos_mistos = df_pagamento.groupby('order_id')['payment_type'].nunique()
print(f"Pedidos com mais de um método de pagamento: {(pedidos_mistos > 1).sum()}")

Pedidos com mais de um método de pagamento: 2246


**Conclusão:** Existem pedidos com múltiplos métodos de pagamento. No modelo dimensional,
`f_payments` terá granularidade de transação — uma linha por evento de pagamento,
não por pedido. `d_payment` armazenará apenas o tipo de pagamento como dimensão.

## 6. Produtos (`olist_products_dataset`)

Contém informações dos produtos: categoria, dimensões e peso.
As categorias estão em português — serão utilizadas diretamente no modelo.

In [25]:
df_produtos = pd.read_csv('../data/raw/olist_products_dataset.csv')
df_produtos.shape

(32951, 9)

In [26]:
df_produtos.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 3.7 MB


In [27]:
df_produtos.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [28]:
# Verificando registros fantasma — product_id presente mas todas as demais colunas nulas
colunas_sem_id = [col for col in df_produtos.columns if col != 'product_id']
registros_fantasma = df_produtos[df_produtos[colunas_sem_id].isna().all(axis=1)]
print(f"Registros fantasma encontrados: {len(registros_fantasma)}")

Registros fantasma encontrados: 1


**Conclusão:** Existem nulos em `product_category_name` e registros fantasma com apenas
`product_id` preenchido. No tratamento, os registros fantasma serão removidos.
Apenas `product_id` e `product_category_name` serão utilizados no modelo.

## 7. Vendedores (`olist_sellers_dataset`)

Contém informações dos vendedores: identificador, cidade e estado.
`seller_state` será utilizado em `f_order_items` via `d_state` — dimensão conformada
compartilhada com o estado do cliente em `f_orders`.

In [29]:
df_vendedor = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
df_vendedor.shape

(3095, 4)

In [30]:
df_vendedor.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 230.3 KB


In [31]:
df_vendedor.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [32]:
df_vendedor.duplicated().sum()

np.int64(0)

In [33]:
df_vendedor.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


**Conclusão:** Sem nulos nem duplicatas. Nenhum tratamento necessário.
`seller_state` será utilizado no modelo para análise geográfica de vendedores.

## Resumo da Exploração

| Tabela | Linhas | Nulos críticos | Ação no tratamento |
|--------|--------|----------------|--------------------|
| customers | 99.441 | Nenhum | Nenhuma |
| orders | 99.441 | Datas de entrega | Converter para datetime |
| order_items | 112.650 | Nenhum | Nenhuma |
| order_reviews | 99.224 | Comentários | Agregar por order_id |
| order_payments | 103.886 | Nenhum | Granularidade de transação |
| products | 32.951 | Categoria e dimensões | Remover registros fantasma |
| sellers | 3.095 | Nenhum | Nenhuma |